In [1]:
pip install selenium webdriver-manager

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install --upgrade typing_extensions

Note: you may need to restart the kernel to use updated packages.


In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

In [10]:
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from datetime import date

def get_film_urls(base_url, num_pages):
    urls = []
    for page in range(1, num_pages + 1):
        driver.get(f"{base_url}page/{page}/")
        time.sleep(3)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        films = soup.find_all("li", class_="poster-container")
        for film in films:
            poster = film.find("div", attrs={"data-film-slug": True})
            if poster:
                slug = poster["data-film-slug"]
                urls.append(f"https://letterboxd.com/film/{slug}/")
        print(f"Page {page}: {len(films)} films found")
    return urls

In [12]:
def scrape_film(url):
    driver.get(url)
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")

    og_title = soup.find("meta", property="og:title")
    title, year = None, None
    if og_title:
        match = re.match(r"^(.*?)\s*\((\d{4})\)$", og_title["content"])
        if match:
            title = match.group(1)
            year = int(match.group(2))

    rating_meta = soup.find("meta", attrs={"name": "twitter:data2"})
    rating = float(rating_meta["content"].split(" out of")[0]) if rating_meta else None

    director_meta = soup.find("meta", attrs={"name": "twitter:data1"})
    director = director_meta["content"] if director_meta else None

    genres = ", ".join([g.text.strip() for g in soup.find_all("a", href=re.compile(r"/films/genre/"))])
    country = ", ".join([c.text.strip() for c in soup.find_all("a", href=re.compile(r"/films/country/"))])
    language = ", ".join(list(set([l.text.strip() for l in soup.find_all("a", href=re.compile(r"/films/language/"))])))

    return {
        "film_title": title,
        "year": year,
        "director": director,
        "avg_rating": rating,
        "genre": genres,
        "country": country,
        "language": language,
        "letterboxd_url": url,
        "date_scraped": str(date.today())
    }

In [14]:
top250_urls = get_film_urls("https://letterboxd.com/films/top250/", num_pages=4)
popular_urls = get_film_urls("https://letterboxd.com/films/popular/", num_pages=5)
all_urls = list(set(top250_urls + popular_urls))
print(f"Total unique films: {len(all_urls)}")

Page 1: 0 films found
Page 2: 0 films found
Page 3: 0 films found
Page 4: 0 films found
Page 1: 0 films found
Page 2: 0 films found
Page 3: 0 films found
Page 4: 0 films found
Page 5: 0 films found
Total unique films: 0
